#DEMANDAS TI

### CONFIGURAÇÃO E CARREGAMENTO DE DATASET

In [28]:
# ============================================================
# Importa as bibliotecas do Python
# ============================================================

# Importar o pandas
import pandas as pd
import numpy as np

# Importar o LabelEncoder da biblioteca scikit-learn
from sklearn.preprocessing import LabelEncoder

# Importar a função de divisão de treino e teste
from sklearn.model_selection import train_test_split

# Modelos
from sklearn.ensemble import RandomForestClassifier
from xgboost          import XGBClassifier
from sklearn.metrics  import (accuracy_score, precision_score,
                               recall_score, f1_score,
                               classification_report)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from imblearn.over_sampling import SMOTE


In [29]:
# ============================================================
# Carregar o dataset a partir do arquivo Excel
# ============================================================

# url do arquivo no GitHub
url_dataset = "https://raw.githubusercontent.com/gilbertoag2007/machine-learning-demandas-ti/main/DEMANDAS_DOWNSTREAM_V3.xlsx"

# Cria um dataframe com o conteúdo do dataset
colunas_desejadas = ["ID_DEMANDA", "SISTEMA", "TIPO_DEMANDA", "DAT_INIC_DESENV_REAL", "DAT_FIM_DESENV_REAL", "DATA_INICIO_PREVISTA", "DATA_FIM_PREVISTA", "CLASSIFICACAO"]
df_original = pd.read_excel(url_dataset, usecols=colunas_desejadas )




# Lista as 5 primeiras colunas do dataframe.

df_original.head()

,ID_DEMANDA,SISTEMA,TIPO_DEMANDA,DAT_INIC_DESENV_REAL,DAT_FIM_DESENV_REAL,DATA_INICIO_PREVISTA,DATA_FIM_PREVISTA,CLASSIFICACAO
0,22391,CSA,ORIENTAÇÃO,2026-05-15 11:08:57,2026-05-15 12:02:28,15/05/2026,2026-05-18,NO PRAZO
1,22381,DPP,BUG IMPEDITIVO,2026-05-15 16:34:46,2026-05-15 18:24:44,14/05/2026,2026-05-15,NO PRAZO
2,22366,I-SIMP (DPP),BUG NÃO IMPEDITIVO,2026-05-12 14:17:40,2026-05-13 10:13:32,12/05/2026,2026-05-18,NO PRAZO
3,22360,DPP,BUG NÃO IMPEDITIVO,2026-05-11 09:55:40,2026-05-15 12:19:49,11/05/2026,2026-05-15,NO PRAZO
4,22327,SIGAF,MELHORIA PEQUENA,2026-05-11 15:49:19,2026-05-11 16:27:17,08/05/2026,2026-05-14,NO PRAZO


##AJUSTES INICIAIS NO DATAFRAME

In [30]:
# ============================================================
# TÉCNICA: Label Encoding (Codificação de Rótulos)
# ============================================================

# Cria uma cópia do dataframe original para preservá-lo intacto
# Todas as alterações serão feitas apenas no df_ajustado
df_ajustado = df_original.copy()

# Cria uma nova coluna numérica baseada na coluna STATUS_FINAL
# map() substitui cada valor categórico pelo número correspondente
df_ajustado['CLASSIFICACAO_FINAL_NUM'] = df_ajustado['CLASSIFICACAO'].map({
    'ATRASO'         : 1,
    'NO PRAZO': 0
})

# Variavel Target
target = "CLASSIFICACAO_FINAL_NUM"



In [31]:
# ============================================================
# TÉCNICA: One-Hot Encoding
# ============================================================

# Aplicar One-Hot Encoding na coluna SISTEMA
# pd.get_dummies() cria uma coluna binária (0 ou 1) para cada sistema único
# dtype=int garante que os valores sejam inteiros ao invés de booleanos
# Aplicar nas colunas categóricas sem ordem natural
for coluna in ['SISTEMA', 'TIPO_DEMANDA']:
    dummies = pd.get_dummies(df_ajustado[coluna], prefix=coluna, dtype=int)
    df_ajustado = pd.concat([df_ajustado, dummies], axis=1)
    df_ajustado = df_ajustado.drop(columns=[coluna])


In [32]:

# ============================================================
# CONVERTER COLUNAS DE DATA PARA DATETIME
# Necessário para realizar operações matemáticas entre datas
# ============================================================
colunas_data = [
    'DATA_INICIO_PREVISTA',
    'DAT_INIC_DESENV_REAL',
    'DATA_FIM_PREVISTA',
    'DAT_FIM_DESENV_REAL'
]

for coluna in colunas_data:
    df_ajustado[coluna] = pd.to_datetime(
        df_ajustado[coluna], dayfirst=True, errors='coerce'
    )


In [33]:
# ============================================================
# GERAR DATA_REFERENCIA
# Ponto médio entre início e fim — simula o momento de acompanhamento de cada demanda
# ============================================================

# Calcular ponto médio como referência padrão
# usando DATA_INICIO_PREVISTA e DATA_FIM_PREVISTA
df_ajustado['DATA_REFERENCIA'] = (
    df_ajustado['DATA_INICIO_PREVISTA'] +
    (df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_INICIO_PREVISTA']) / 2
)

# Ajustar referência para demandas já iniciadas
# usar ponto médio entre DATA_INICIO_REALIZADA e DATA_FIM_PREVISTA
mask_iniciadas = df_ajustado['DAT_INIC_DESENV_REAL'].notna()

df_ajustado.loc[mask_iniciadas, 'DATA_REFERENCIA'] = (
    df_ajustado.loc[mask_iniciadas, 'DAT_INIC_DESENV_REAL'] +
    (
        df_ajustado.loc[mask_iniciadas, 'DATA_FIM_PREVISTA'] -
        df_ajustado.loc[mask_iniciadas, 'DAT_INIC_DESENV_REAL']
    ) / 2
)

In [34]:
df_ajustado.head(50)

,ID_DEMANDA,DAT_INIC_DESENV_REAL,DAT_FIM_DESENV_REAL,DATA_INICIO_PREVISTA,DATA_FIM_PREVISTA,CLASSIFICACAO,CLASSIFICACAO_FINAL_NUM,SISTEMA_CSA,SISTEMA_DFe - Documento de Fiscalização Eletrônico,SISTEMA_DPP,...,SISTEMA_SIGAF,SISTEMA_SIMP,SISTEMA_SRD - GLP,SISTEMA_SRD - PR,TIPO_DEMANDA_BUG IMPEDITIVO,TIPO_DEMANDA_BUG NÃO IMPEDITIVO,TIPO_DEMANDA_MELHORIA MÉDIA,TIPO_DEMANDA_MELHORIA PEQUENA,TIPO_DEMANDA_ORIENTAÇÃO,DATA_REFERENCIA
0,22391,2026-05-15 11:08:57,2026-05-15 12:02:28,2026-05-15,2026-05-18,NO PRAZO,0,1,0,0,...,0,0,0,0,0,0,0,0,1,2026-05-16 17:34:28.500
1,22381,2026-05-15 16:34:46,2026-05-15 18:24:44,2026-05-14,2026-05-15,NO PRAZO,0,0,0,1,...,0,0,0,0,1,0,0,0,0,2026-05-15 08:17:23.000
2,22366,2026-05-12 14:17:40,2026-05-13 10:13:32,2026-05-12,2026-05-18,NO PRAZO,0,0,0,0,...,0,0,0,0,0,1,0,0,0,2026-05-15 07:08:50.000
3,22360,2026-05-11 09:55:40,2026-05-15 12:19:49,2026-05-11,2026-05-15,NO PRAZO,0,0,0,1,...,0,0,0,0,0,1,0,0,0,2026-05-13 04:57:50.000
4,22327,2026-05-11 15:49:19,2026-05-11 16:27:17,2026-05-08,2026-05-14,NO PRAZO,0,0,0,0,...,1,0,0,0,0,0,0,1,0,2026-05-12 19:54:39.500
5,22320,2026-05-04 14:13:17,2026-05-04 14:42:27,2026-05-07,2026-05-13,NO PRAZO,0,0,0,0,...,0,0,0,0,0,0,0,1,0,2026-05-08 19:06:38.500
6,22312,2026-05-05 10:23:14,2026-05-05 16:20:32,2026-05-04,2026-05-05,NO PRAZO,0,0,1,0,...,0,0,0,0,1,0,0,0,0,2026-05-05 05:11:37.000
7,22307,2026-04-30 19:04:17,2026-04-30 19:06:11,2026-04-30,2026-05-04,NO PRAZO,0,0,0,1,...,0,0,0,0,0,0,0,0,1,2026-05-02 09:32:08.500
8,22302,2026-05-04 15:35:23,2026-05-05 16:21:17,2026-04-30,2026-05-07,NO PRAZO,0,1,0,0,...,0,0,0,0,0,1,0,0,0,2026-05-05 19:47:41.500
9,22298,2026-04-30 15:50:24,2026-05-07 15:04:55,2026-04-30,2026-05-07,NO PRAZO,0,1,0,0,...,0,0,0,0,0,1,0,0,0,2026-05-03 19:55:12.000


In [35]:
# ============================================================
# GERAR FEATURES NUMÉRICAS
# ============================================================

# ----------------------------------------------------------
# Duração total planejada da demanda em dias
# Indica o tamanho e complexidade da demanda
# ----------------------------------------------------------
df_ajustado['DURACAO_PREVISTA_DIAS'] = (
    df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days

# ----------------------------------------------------------
# Quantidade de dias de atraso no início da demanda
# Valores positivos indicam atraso no início
# Preenchido com 0 quando não há DATA_INICIO_REALIZADA
# ----------------------------------------------------------
df_ajustado['ATRASO_INICIO_DIAS'] = (
    df_ajustado['DAT_INIC_DESENV_REAL'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days.fillna(0)

# ----------------------------------------------------------
# Dias restantes até o prazo final na data de referência
# Valores negativos indicam que o prazo já foi ultrapassado
# ----------------------------------------------------------
df_ajustado['DIAS_RESTANTES'] = (
    df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_REFERENCIA']
).dt.days

# ----------------------------------------------------------
# Percentual do prazo consumido até a data de referência
# Indica o quanto do tempo planejado já foi utilizado
# ----------------------------------------------------------
df_ajustado['PERC_TEMPO_DECORRIDO'] = (
    (df_ajustado['DATA_REFERENCIA'] - df_ajustado['DATA_INICIO_PREVISTA']).dt.days /
     df_ajustado['DURACAO_PREVISTA_DIAS']
) * 100

# ----------------------------------------------------------
# Dias sem iniciar após a DATA_INICIO_PREVISTA
# clip(lower=0) evita valores negativos para demandas
# que ainda não atingiram a data de início prevista
# ----------------------------------------------------------
df_ajustado['DIAS_SEM_INICIAR'] = (
    df_ajustado['DATA_REFERENCIA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days.clip(lower=0)


In [36]:
df_ajustado.head(50)

,ID_DEMANDA,DAT_INIC_DESENV_REAL,DAT_FIM_DESENV_REAL,DATA_INICIO_PREVISTA,DATA_FIM_PREVISTA,CLASSIFICACAO,CLASSIFICACAO_FINAL_NUM,SISTEMA_CSA,SISTEMA_DFe - Documento de Fiscalização Eletrônico,SISTEMA_DPP,...,TIPO_DEMANDA_BUG NÃO IMPEDITIVO,TIPO_DEMANDA_MELHORIA MÉDIA,TIPO_DEMANDA_MELHORIA PEQUENA,TIPO_DEMANDA_ORIENTAÇÃO,DATA_REFERENCIA,DURACAO_PREVISTA_DIAS,ATRASO_INICIO_DIAS,DIAS_RESTANTES,PERC_TEMPO_DECORRIDO,DIAS_SEM_INICIAR
0,22391,2026-05-15 11:08:57,2026-05-15 12:02:28,2026-05-15,2026-05-18,NO PRAZO,0,1,0,0,...,0,0,0,1,2026-05-16 17:34:28.500,3,0,1,33.333333,1
1,22381,2026-05-15 16:34:46,2026-05-15 18:24:44,2026-05-14,2026-05-15,NO PRAZO,0,0,0,1,...,0,0,0,0,2026-05-15 08:17:23.000,1,1,-1,100.000000,1
2,22366,2026-05-12 14:17:40,2026-05-13 10:13:32,2026-05-12,2026-05-18,NO PRAZO,0,0,0,0,...,1,0,0,0,2026-05-15 07:08:50.000,6,0,2,50.000000,3
3,22360,2026-05-11 09:55:40,2026-05-15 12:19:49,2026-05-11,2026-05-15,NO PRAZO,0,0,0,1,...,1,0,0,0,2026-05-13 04:57:50.000,4,0,1,50.000000,2
4,22327,2026-05-11 15:49:19,2026-05-11 16:27:17,2026-05-08,2026-05-14,NO PRAZO,0,0,0,0,...,0,0,1,0,2026-05-12 19:54:39.500,6,3,1,66.666667,4
5,22320,2026-05-04 14:13:17,2026-05-04 14:42:27,2026-05-07,2026-05-13,NO PRAZO,0,0,0,0,...,0,0,1,0,2026-05-08 19:06:38.500,6,-3,4,16.666667,1
6,22312,2026-05-05 10:23:14,2026-05-05 16:20:32,2026-05-04,2026-05-05,NO PRAZO,0,0,1,0,...,0,0,0,0,2026-05-05 05:11:37.000,1,1,-1,100.000000,1
7,22307,2026-04-30 19:04:17,2026-04-30 19:06:11,2026-04-30,2026-05-04,NO PRAZO,0,0,0,1,...,0,0,0,1,2026-05-02 09:32:08.500,4,0,1,50.000000,2
8,22302,2026-05-04 15:35:23,2026-05-05 16:21:17,2026-04-30,2026-05-07,NO PRAZO,0,1,0,0,...,1,0,0,0,2026-05-05 19:47:41.500,7,4,1,71.428571,5
9,22298,2026-04-30 15:50:24,2026-05-07 15:04:55,2026-04-30,2026-05-07,NO PRAZO,0,1,0,0,...,1,0,0,0,2026-05-03 19:55:12.000,7,0,3,42.857143,3


In [37]:
# ============================================================
# GERAR FLAGS BINÁRIAS (0 ou 1)
# ============================================================

# ----------------------------------------------------------
# Flag: a demanda já foi iniciada?
# 1 = sim | 0 = não
# ----------------------------------------------------------
df_ajustado['FLAG_INICIADA'] = (
    df_ajustado['DAT_INIC_DESENV_REAL'].notna()
).astype(int)

# ----------------------------------------------------------
# Flag: a demanda atrasou no início?
# 1 = começou depois do previsto | 0 = não
# ----------------------------------------------------------
df_ajustado['FLAG_ATRASO_INICIO'] = (
    df_ajustado['ATRASO_INICIO_DIAS'] > 0
).astype(int)

# ----------------------------------------------------------
# Flag: a demanda deveria ter iniciado mas ainda não iniciou?
# 1 = passou da DATA_INICIO_PREVISTA sem início registrado
# 0 = ainda dentro do prazo de início ou já iniciada
# ----------------------------------------------------------
df_ajustado['FLAG_NAO_INICIADA_NO_PRAZO'] = (
    (df_ajustado['DATA_REFERENCIA'] >= df_ajustado['DATA_INICIO_PREVISTA']) &
    (df_ajustado['DAT_INIC_DESENV_REAL'].isna())
).astype(int)

In [38]:
# ============================================================
# REMOVER COLUNAS QUE NÃO DEVEM ENTRAR NO MODELO ANTES DO TREINAMENTO
# ============================================================

colunas_remover = [
    'ID_DEMANDA',
    'CLASSIFICACAO',
    'DATA_INICIO_PREVISTA',
    'DAT_INIC_DESENV_REAL',
    'DATA_FIM_PREVISTA',
    'DAT_FIM_DESENV_REAL',
    'DATA_REFERENCIA'
]
df_ajustado = df_ajustado.drop(columns=colunas_remover)

In [39]:
# Exibir resumo das colunas geradas e seus tipos
print('📊 Colunas do dataframe ajustado:')
print(df_ajustado.dtypes)
print(f'\n✅ Shape final: {df_ajustado.shape[0]} linhas x {df_ajustado.shape[1]} colunas')

df_ajustado.head(50)

📊 Colunas do dataframe ajustado:
CLASSIFICACAO_FINAL_NUM                                 int64
SISTEMA_CSA                                             int64
SISTEMA_DFe - Documento de Fiscalização Eletrônico      int64
SISTEMA_DPP                                             int64
SISTEMA_I-SIMP (DPP)                                    int64
SISTEMA_RENOVACALC                                      int64
SISTEMA_SIGAF                                           int64
SISTEMA_SIMP                                            int64
SISTEMA_SRD - GLP                                       int64
SISTEMA_SRD - PR                                        int64
TIPO_DEMANDA_BUG IMPEDITIVO                             int64
TIPO_DEMANDA_BUG NÃO IMPEDITIVO                         int64
TIPO_DEMANDA_MELHORIA MÉDIA                             int64
TIPO_DEMANDA_MELHORIA PEQUENA                           int64
TIPO_DEMANDA_ORIENTAÇÃO                                 int64
DURACAO_PREVISTA_DIAS                

,CLASSIFICACAO_FINAL_NUM,SISTEMA_CSA,SISTEMA_DFe - Documento de Fiscalização Eletrônico,SISTEMA_DPP,SISTEMA_I-SIMP (DPP),SISTEMA_RENOVACALC,SISTEMA_SIGAF,SISTEMA_SIMP,SISTEMA_SRD - GLP,SISTEMA_SRD - PR,...,TIPO_DEMANDA_MELHORIA PEQUENA,TIPO_DEMANDA_ORIENTAÇÃO,DURACAO_PREVISTA_DIAS,ATRASO_INICIO_DIAS,DIAS_RESTANTES,PERC_TEMPO_DECORRIDO,DIAS_SEM_INICIAR,FLAG_INICIADA,FLAG_ATRASO_INICIO,FLAG_NAO_INICIADA_NO_PRAZO
0,0,1,0,0,0,0,0,0,0,0,...,0,1,3,0,1,33.333333,1,1,0,0
1,0,0,0,1,0,0,0,0,0,0,...,0,0,1,1,-1,100.000000,1,1,1,0
2,0,0,0,0,1,0,0,0,0,0,...,0,0,6,0,2,50.000000,3,1,0,0
3,0,0,0,1,0,0,0,0,0,0,...,0,0,4,0,1,50.000000,2,1,0,0
4,0,0,0,0,0,0,1,0,0,0,...,1,0,6,3,1,66.666667,4,1,1,0
5,0,0,0,0,1,0,0,0,0,0,...,1,0,6,-3,4,16.666667,1,1,0,0
6,0,0,1,0,0,0,0,0,0,0,...,0,0,1,1,-1,100.000000,1,1,1,0
7,0,0,0,1,0,0,0,0,0,0,...,0,1,4,0,1,50.000000,2,1,0,0
8,0,1,0,0,0,0,0,0,0,0,...,0,0,7,4,1,71.428571,5,1,1,0
9,0,1,0,0,0,0,0,0,0,0,...,0,0,7,0,3,42.857143,3,1,0,0


#ANALISE DOS DADOS

In [40]:
# ============================================================
# VERIFICAR O BALANCEAMENTO DA COLUNA TARGET
# ============================================================

balanceamento = df_ajustado[target].value_counts()
percentual    = df_ajustado[target].value_counts(normalize=True) * 100

# Exibir resultado
print('Distribuição da variável target:\n')
print(f'🟢 DENTRO DO PRAZO (0): {balanceamento[0]} registros ({percentual[0]:.1f}%)')
print(f'🔴 ATRASO          (1): {balanceamento[1]} registros ({percentual[1]:.1f}%)')

Distribuição da variável target:

🟢 DENTRO DO PRAZO (0): 1456 registros (89.2%)
🔴 ATRASO          (1): 176 registros (10.8%)


#TESTANDO OS MODELOS

In [41]:

# ----------------------------------------------------------
# Separar features (X) da variável target (y)
# X = colunas que o modelo usa para aprender
# y = coluna que o modelo deve prever
# ----------------------------------------------------------


X = df_ajustado.drop(columns=[target])
y = df_ajustado[target]

# ----------------------------------------------------------
# Dividir em treino (70%) e teste (30%)
# stratify=y garante que a proporção de 0 e 1 seja mantida
# igual nos dois conjuntos — essencial para dados desbalanceados
# random_state=42 garante que a divisão seja reproduzível
# ----------------------------------------------------------
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size    = 0.30,
    stratify     = y,
    random_state = 42
)

# Exibir o resultado da divisão
print(f'Total de registros  : {len(X)}')
print(f'Registros de treino : {len(X_treino)} ({len(X_treino)/len(X)*100:.1f}%)')
print(f'Registros de teste  : {len(X_teste)} ({len(X_teste)/len(X)*100:.1f}%)')
print(f'\nDistribuição do target no treino:\n{y_treino.value_counts()}')
print(f'\nDistribuição do target no teste:\n{y_teste.value_counts()}')

Total de registros  : 1632
Registros de treino : 1142 (70.0%)
Registros de teste  : 490 (30.0%)

Distribuição do target no treino:
CLASSIFICACAO_FINAL_NUM
0    1019
1     123
Name: count, dtype: int64

Distribuição do target no teste:
CLASSIFICACAO_FINAL_NUM
0    437
1     53
Name: count, dtype: int64


In [42]:
# ─────────────────────────────────────────────
# 2. SMOTE — cria amostras sintéticas da classe minoritária
#    Só aplicar no conjunto de TREINO, nunca no teste!
# ─────────────────────────────────────────────
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_treino, y_treino)

print(f"Antes do SMOTE : {y_treino.value_counts().to_dict()}")
print(f"Depois do SMOTE: {y_train_bal.value_counts().to_dict()}")

Antes do SMOTE : {0: 1019, 1: 123}
Depois do SMOTE: {1: 1019, 0: 1019}


In [43]:

# ============================================================
# TREINAR O MODELO RANDOM FOREST
# ============================================================

rf_modelo = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced_subsample',
    min_samples_split=10,
    min_samples_leaf=2,
    max_features='log2',
    max_depth=None,
    random_state=42
)

rf_modelo.fit(X_train_bal, y_train_bal)

# ─────────────────────────────────────────────
# MUDANÇA 1: trocar predict() por predict_proba()
# predict()      → retorna direto 0 ou 1 (usa threshold 0.5 fixo)
# predict_proba() → retorna a probabilidade de cada classe
# ─────────────────────────────────────────────
rf_probabilidades = rf_modelo.predict_proba(X_teste)[:, 1]
# [:, 1] pega só a coluna da classe "Atraso" (classe positiva)


# ─────────────────────────────────────────────
# MUDANÇA 2: aplicar o threshold manualmente
# Qualquer probabilidade >= threshold vira "Atraso"
# Reduza o valor para capturar mais atrasos (mais Recall)
# ─────────────────────────────────────────────
threshold = 0.40  # ← ajuste esse valor (padrão seria 0.50)

rf_predicao = (rf_probabilidades >= threshold).astype(int)


# ─────────────────────────────────────────────
# EXTRA: testar vários thresholds de uma vez
# para encontrar o melhor ponto de equilíbrio
# ─────────────────────────────────────────────
for t in [0.45, 0.40, 0.35, 0.30, 0.25, 0.20]:
    predicao_t = (rf_probabilidades >= t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_teste, predicao_t,
                                target_names=["Dentro do Prazo", "Atraso"]))




Threshold: 0.45
                 precision    recall  f1-score   support

Dentro do Prazo       0.96      0.93      0.94       437
         Atraso       0.52      0.66      0.58        53

       accuracy                           0.90       490
      macro avg       0.74      0.79      0.76       490
   weighted avg       0.91      0.90      0.90       490


Threshold: 0.4
                 precision    recall  f1-score   support

Dentro do Prazo       0.96      0.92      0.94       437
         Atraso       0.51      0.66      0.57        53

       accuracy                           0.89       490
      macro avg       0.73      0.79      0.76       490
   weighted avg       0.91      0.89      0.90       490


Threshold: 0.35
                 precision    recall  f1-score   support

Dentro do Prazo       0.96      0.88      0.92       437
         Atraso       0.42      0.70      0.52        53

       accuracy                           0.86       490
      macro avg       0.69    

In [44]:
# ============================================================
# TREINAR O NODELO XGBOOST
# ============================================================

# Calcular o peso das classes para lidar com desbalanceamento
# scale_pos_weight = total de negativos / total de positivos
escala_peso = (y_treino == 0).sum() / (y_treino == 1).sum()

# Instanciar o modelo XGBoost
# scale_pos_weight ajusta o peso das classes desbalanceadas
# random_state=42 garante reprodutibilidade dos resultados
xgb_modelo = XGBClassifier(
    subsample= 0.8,
    n_estimators=200,
    min_child_weight=3,
    scale_pos_weight = np.float64(14.235294117647058),
    random_state= 42,
    max_depth=3,
    colsample_bytree=0.8


    #eval_metric      = 'logloss'
)

# Treinar o modelo com os dados de treino
#xgb_modelo.fit(X_treino, y_treino)
xgb_modelo.fit(X_train_bal, y_train_bal) #smote

# Gerar predições com os dados de teste
xgb_predicao = xgb_modelo.predict(X_teste)


 Recall e o F1-Score são as métricas mais importantes — é melhor o modelo alertar um possível atraso que não vai acontecer do que deixar passar um atraso real sem aviso.

In [45]:
# ============================================================
# AVALIAR E COMPARAR OS MODELOS
# ============================================================

# Função para calcular e exibir as métricas de cada modelo
def avaliar_modelo(nome, y_teste, y_predicao):
    print(f'\n{"="*50}')
    print(f'  {nome}')
    print(f'{"="*50}')
    print(f'Acurácia  : {accuracy_score(y_teste, y_predicao):.2%}')
    print(f'Precisão  : {precision_score(y_teste, y_predicao):.2%}')
    print(f'Recall    : {recall_score(y_teste, y_predicao):.2%}')
    print(f'F1-Score  : {f1_score(y_teste, y_predicao):.2%}')
    print(f'\nRelatório completo:')
    print(classification_report(y_teste, y_predicao,
                                target_names=['Dentro do Prazo', 'Atraso']))

# Avaliar os dois modelos
avaliar_modelo('RANDOM FOREST', y_teste, rf_predicao)
avaliar_modelo('XGBOOST'      , y_teste, xgb_predicao)


  RANDOM FOREST
Acurácia  : 89.39%
Precisão  : 50.72%
Recall    : 66.04%
F1-Score  : 57.38%

Relatório completo:
                 precision    recall  f1-score   support

Dentro do Prazo       0.96      0.92      0.94       437
         Atraso       0.51      0.66      0.57        53

       accuracy                           0.89       490
      macro avg       0.73      0.79      0.76       490
   weighted avg       0.91      0.89      0.90       490


  XGBOOST
Acurácia  : 82.45%
Precisão  : 33.66%
Recall    : 64.15%
F1-Score  : 44.16%

Relatório completo:
                 precision    recall  f1-score   support

Dentro do Prazo       0.95      0.85      0.90       437
         Atraso       0.34      0.64      0.44        53

       accuracy                           0.82       490
      macro avg       0.64      0.74      0.67       490
   weighted avg       0.88      0.82      0.85       490



In [46]:
# ============================================================
# OTIMIZAÇÃO DE HIPERPARÂMETROS
# Técnica: RandomizedSearchCV
# Mais eficiente que GridSearchCV pois testa combinações
# aleatórias ao invés de todas as combinações possíveis
# ============================================================


# ============================================================
# HIPERPARÂMETROS DO RANDOM FOREST
# ============================================================

# Definir o grid de hiperparâmetros a testar
rf_parametros = {

    # Quantidade de árvores de decisão
    # Mais árvores = mais robusto, porém mais lento
    'n_estimators': [100, 200, 300, 500],

    # Profundidade máxima de cada árvore
    # None = cresce até separar todas as classes (risco de overfitting)
    # Valores menores = modelo mais simples e generalista
    'max_depth': [None, 5, 10, 20, 30],

    # Mínimo de amostras para dividir um nó interno
    # Valores maiores = árvores mais simples, menos overfitting
    'min_samples_split': [2, 5, 10],

    # Mínimo de amostras em um nó folha
    # Valores maiores = modelo mais conservador
    'min_samples_leaf': [1, 2, 4],

    # Quantidade de features consideradas em cada divisão
    # sqrt = raiz quadrada do total de features (padrão)
    # log2 = logaritmo base 2 do total de features
    'max_features': ['sqrt', 'log2'],

    # Peso das classes para lidar com desbalanceamento
    'class_weight': ['balanced', 'balanced_subsample']
}

# Instanciar o modelo base
rf_modelo_otimizado = RandomForestClassifier(random_state=42)

# Instanciar o RandomizedSearchCV
# n_iter=50 testa 50 combinações aleatórias do grid
# cv=5 usa validação cruzada com 5 divisões
# scoring='f1' otimiza pelo F1-Score — ideal para dados desbalanceados
# n_jobs=-1 usa todos os núcleos do processador para acelerar
rf_busca = RandomizedSearchCV(
    estimator          = rf_modelo_otimizado,
    param_distributions = rf_parametros,
    n_iter             = 50,
    cv                 = 5,
    scoring            = 'f1',
    n_jobs             = -1,
    random_state       = 42,
    verbose            = 1
)

# Treinar com os dados de treino
rf_busca.fit(X_treino, y_treino)

# Exibir os melhores hiperparâmetros encontrados
print('Melhores hiperparâmetros — Random Forest:')
print(rf_busca.best_params_)

# ============================================================
# BLOCO 2 — HIPERPARÂMETROS DO XGBOOST
# ============================================================

xgb_parametros = {

    # Quantidade de árvores (rodadas de boosting)
    'n_estimators': [100, 200, 300, 500],

    # Profundidade máxima de cada árvore
    # Valores entre 3 e 10 são os mais comuns
    'max_depth': [3, 5, 7, 10],

    # Taxa de aprendizado — controla o peso de cada árvore
    # Valores menores = aprendizado mais lento e preciso
    'learning_rate': [0.01, 0.05, 0.1, 0.2],

    # Proporção de features usadas por árvore
    # Reduz overfitting ao não usar todas as features sempre
    'colsample_bytree': [0.6, 0.8, 1.0],

    # Proporção de amostras usadas por árvore
    # Reduz overfitting ao não usar todos os registros sempre
    'subsample': [0.6, 0.8, 1.0],

    # Peso mínimo necessário para criar um novo nó folha
    # Valores maiores = modelo mais conservador
    'min_child_weight': [1, 3, 5],

    # Peso das classes desbalanceadas
    # Calculado como: total negativos / total positivos
    'scale_pos_weight': [
        (y_treino == 0).sum() / (y_treino == 1).sum()
    ]
}

# Instanciar o modelo base
xgb_modelo_otimizado = XGBClassifier(
    random_state = 42,
    eval_metric  = 'logloss'
)

# Instanciar o RandomizedSearchCV para o XGBoost
xgb_busca = RandomizedSearchCV(
    estimator           = xgb_modelo_otimizado,
    param_distributions = xgb_parametros,
    n_iter              = 50,
    cv                  = 5,
    scoring             = 'f1',
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 1
)

# Treinar com os dados de treino
xgb_busca.fit(X_treino, y_treino)

# Exibir os melhores hiperparâmetros encontrados
print('\nMelhores hiperparâmetros — XGBoost:')
print(xgb_busca.best_params_)

# ============================================================
# BLOCO 3 — AVALIAR OS MODELOS OTIMIZADOS
# ============================================================

# Gerar predições com os melhores modelos encontrados
rf_melhor_predicao  = rf_busca.best_estimator_.predict(X_teste)
xgb_melhor_predicao = xgb_busca.best_estimator_.predict(X_teste)

# Avaliar os dois modelos otimizados
avaliar_modelo('RANDOM FOREST OTIMIZADO', y_teste, rf_melhor_predicao)
avaliar_modelo('XGBOOST OTIMIZADO'      , y_teste, xgb_melhor_predicao)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Melhores hiperparâmetros — Random Forest:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 10, 'class_weight': 'balanced'}
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Melhores hiperparâmetros — XGBoost:
{'subsample': 0.8, 'scale_pos_weight': np.float64(8.284552845528456), 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}

  RANDOM FOREST OTIMIZADO
Acurácia  : 93.88%
Precisão  : 75.56%
Recall    : 64.15%
F1-Score  : 69.39%

Relatório completo:
                 precision    recall  f1-score   support

Dentro do Prazo       0.96      0.97      0.97       437
         Atraso       0.76      0.64      0.69        53

       accuracy                           0.94       490
      macro avg       0.86      0.81      0.83       490
   weighted avg       0.94      0.94      0.94       490


  XGBOOST O

In [49]:
# ============================================================
# PREDIÇÃO DE NOVO REGISTRO
# ============================================================


# ----------------------------------------------------------
# Preencher os dados da nova demanda
# Altere os valores conforme a demanda que deseja testar
# ----------------------------------------------------------
nova_demanda = {
    'ID_DEMANDA'            : 'DEM-999',
    'SISTEMA'               : 'DFE',
    'TIPO_DEMANDA'          : 'MELHORIA_MEDIA',
    'DATA_INICIO_PREVISTA'  : '01/05/2025',
    'DAT_INIC_DESENV_REAL' : '11/05/2025',            # None se ainda não iniciada
    'DATA_FIM_PREVISTA'     : '21/05/2025',
    'DATA_REFERENCIA'       : '15/05/2025'     # data do momento da consulta
}

df_nova = pd.DataFrame([nova_demanda])

# ============================================================
# BLOCO 1 — CONVERTER DATAS PARA DATETIME
# ============================================================
colunas_data = [
    'DATA_INICIO_PREVISTA',
    'DAT_INIC_DESENV_REAL',
    'DATA_FIM_PREVISTA',
    'DATA_REFERENCIA'
]

for coluna in colunas_data:
    df_nova[coluna] = pd.to_datetime(
        df_nova[coluna], dayfirst=True, errors='coerce'
    )

# ============================================================
# BLOCO 2 — GERAR AS FEATURES
# Mesma lógica aplicada no treinamento — obrigatório
# ============================================================

df_nova['DURACAO_PREVISTA_DIAS'] = (
    df_nova['DATA_FIM_PREVISTA'] - df_nova['DATA_INICIO_PREVISTA']
).dt.days

df_nova['ATRASO_INICIO_DIAS'] = (
    df_nova['DAT_INIC_DESENV_REAL'] - df_nova['DATA_INICIO_PREVISTA']
).dt.days.fillna(0)

df_nova['DIAS_RESTANTES'] = (
    df_nova['DATA_FIM_PREVISTA'] - df_nova['DATA_REFERENCIA']
).dt.days

df_nova['PERC_TEMPO_DECORRIDO'] = (
    (df_nova['DATA_REFERENCIA'] - df_nova['DATA_INICIO_PREVISTA']).dt.days /
     df_nova['DURACAO_PREVISTA_DIAS']
) * 100

df_nova['DIAS_SEM_INICIAR'] = (
    df_nova['DATA_REFERENCIA'] - df_nova['DATA_INICIO_PREVISTA']
).dt.days.clip(lower=0)

df_nova['FLAG_INICIADA'] = (
    df_nova['DAT_INIC_DESENV_REAL'].notna()
).astype(int)

df_nova['FLAG_ATRASO_INICIO'] = (
    df_nova['ATRASO_INICIO_DIAS'] > 0
).astype(int)

df_nova['FLAG_NAO_INICIADA_NO_PRAZO'] = (
    (df_nova['DATA_REFERENCIA'] >= df_nova['DATA_INICIO_PREVISTA']) &
    (df_nova['DAT_INIC_DESENV_REAL'].isna())
).astype(int)

# ============================================================
# BLOCO 3 — APLICAR ONE-HOT ENCODING
# Usar get_dummies e alinhar com as colunas do treino
# para garantir que o modelo receba as mesmas colunas
# ============================================================

# Aplicar One-Hot Encoding nas colunas categóricas
df_nova = pd.get_dummies(df_nova, columns=['SISTEMA', 'TIPO_DEMANDA'], dtype=int)

# Alinhar colunas com o dataset de treino
# Colunas ausentes são preenchidas com 0
# Colunas extras são removidas
df_nova = df_nova.reindex(columns=X_treino.columns, fill_value=0)

# ============================================================
# BLOCO 4 — REMOVER COLUNAS QUE NÃO ENTRAM NO MODELO
# ============================================================
colunas_remover = [
    'ID_DEMANDA',
    'STATUS_FINAL',
    'DATA_INICIO_PREVISTA',
    'DAT_INIC_DESENV_REAL',
    'DATA_FIM_PREVISTA',
    'DAT_FIM_DESENV_REAL',
    'DATA_REFERENCIA'
]

# Remover apenas as colunas que existirem no dataframe
colunas_existentes = [c for c in colunas_remover if c in df_nova.columns]
df_nova = df_nova.drop(columns=colunas_existentes)

# ============================================================
# BLOCO 5 — GERAR PREDIÇÕES E PROBABILIDADES
# ============================================================

# Função para exibir o resultado da predição
def exibir_predicao(nome_modelo, modelo, dados):

    # Predição binária — 0 ou 1
    predicao = modelo.predict(dados)[0]

    # Probabilidade de cada classe em percentual
    probabilidade = modelo.predict_proba(dados)[0]

    resultado = '🔴 RISCO DE ATRASO' if predicao == 1 else '🟢 DENTRO DO PRAZO'

    print(f'\n{"="*50}')
    print(f'  {nome_modelo}')
    print(f'{"="*50}')
    print(f'Resultado         : {resultado}')
    print(f'Prob. No Prazo    : {probabilidade[0]:.2%}')
    print(f'Prob. Atraso      : {probabilidade[1]:.2%}')

# Exibir predição dos dois modelos
exibir_predicao('RANDOM FOREST OTIMIZADO', rf_busca.best_estimator_,  df_nova)
exibir_predicao('XGBOOST OTIMIZADO'      , xgb_busca.best_estimator_, df_nova)


  RANDOM FOREST OTIMIZADO
Resultado         : 🟢 DENTRO DO PRAZO
Prob. No Prazo    : 58.07%
Prob. Atraso      : 41.93%

  XGBOOST OTIMIZADO
Resultado         : 🔴 RISCO DE ATRASO
Prob. No Prazo    : 43.80%
Prob. Atraso      : 56.20%
